# USD Rates RV v2 — Rolling PCA fly screener

Showcases the v2 RV toolkit additions on ~1y EOD SOFR data:
- **Rolling PCA residual** with eigenvector continuity (`rolling_residual`, `align_eigenvectors`)
- **Optimal OU bands** — Zeng-Lee 2014 (`optimal_ou_thresholds`)
- **OU S-score** — Avellaneda-Lee (`ou_sscore`)
- **ADF gate** (`adf_gate`)
- **Eigenportfolio returns** (`eigenportfolio_returns`)
- **Lead-lag** — Lévy area + cross-correlation (`levy_area`, `xcorr_lead_lag`)
- **Cost-aware screener** (`make_pca_fly_rv_screener`)

Mirrors `rv_pca_curve_fly.ipynb` plumbing.

In [ ]:
%matplotlib inline
import sys; sys.path.append("../../")
import datetime, pytz, warnings
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
plt.rcParams["figure.figsize"] = (15, 6); plt.rcParams["axes.grid"] = True
from RVUtils.plt_timeseries import make_secondary_axis_plot

NYC = pytz.timezone("America/New_York")
from MDP.IRSwaps.IRSwapsMDP import IRSwapsMDP
from TB.IRSwapsTB import IRSwapsTB
from MDP.FixedRateBonds.FixedRateBondsMDP import FixedRateBondsMDP
from TB.FixedRateBondsTB import FixedRateBondsTB
from Query.Unified.UnifiedQuery import UnifiedQuery
from Query.Unified.registry import UnifiedValue
from TB.TimeseriesBuilder import TimeseriesBuilder

curve_mdp = IRSwapsMDP(source="ERIS_EOD_LIVE-RL_BASIC")
usts_mdp = FixedRateBondsMDP(source="USTS_FEDINVEST_WSJ_LIVE-RL")
ts = TimeseriesBuilder()
start = NYC.localize(datetime.datetime(2025, 6, 2, 17, 0))
end = NYC.localize(datetime.datetime(2026, 6, 24, 17, 0))
ROUTERS = {"IRS": IRSwapsTB(curve_mdp, show_tqdm=False), "FRB": FixedRateBondsTB(usts_mdp, show_tqdm=False)}

def load(queries):
    return ts.get_timeseries(start=start, end=end, queries=queries, n_jobs=8, routers=ROUTERS)

def sofr(tenors):
    df = load([UnifiedQuery(curve="USD-SOFR-1D", tenor=t, value=UnifiedValue.IRS_RATE) for t in tenors])
    return df.rename(columns={f"USD-SOFR-1D {t} OUTRIGHT RATE": t for t in tenors})[list(tenors)]

print("Setup OK. Window:", start.date(), "->", end.date())

In [ ]:
TENORS = ["2y", "3y", "5y", "7y", "10y", "20y", "30y"]
df_s = sofr(TENORS)
print(f"SOFR curve: {df_s.shape[0]} rows x {df_s.shape[1]} tenors")
df_s.tail(3)

## 1. Rolling PCA residual with eigenvector continuity (spec A)

Re-fits PCA on a trailing 261-day window at each step. `align_eigenvectors` flips sign of eigenvectors
across windows to prevent spurious jumps in the residual.

In [ ]:
from RVUtils.pca_rv import rolling_residual, make_pca_rv_builder, eigenportfolio_returns

# Rolling PCA residual for outright 10y
resid_10y = rolling_residual(df_s, "10y", window=200, k=3, sign_align=True)
print(f"Rolling residual: {resid_10y.notna().sum()} observations")

plot, fig, ax, ax2, legend = make_secondary_axis_plot(
    engine="matplotlib", title="10y SOFR: rolling PCA residual (200d window, k=3)"
)
plot(resid_10y.rename("10y rolling PCA residual") * 100, which="left")
z = (resid_10y - resid_10y.rolling(65).mean()) / resid_10y.rolling(65).std()
plot(z.rename("z-score (65d)"), which="right",
     indicators=[{"kind": "zbands", "entry": 2, "stop": 3}])
legend(show_date=True, loc="upper left")
plt.show()

In [ ]:
# Rolling PCA residual for 2s5s10s fly
fit_fn, fv_fn, res_fn, fly_w_fn, *_ = make_pca_rv_builder(df_s, on="levels", n_factors=3, sort_by_tenor=False)
fit_fn()
w_2s5s10s = fly_w_fn("2y", "5y", "10y")
print("PCA-neutral 2s5s10s weights:", {k: round(v, 3) for k, v in w_2s5s10s.items()})

resid_fly = rolling_residual(df_s, ["2y", "5y", "10y"], weights=w_2s5s10s, window=200, k=3)

plot, fig, ax, ax2, legend = make_secondary_axis_plot(
    engine="matplotlib", title="2s5s10s SOFR fly: rolling PCA residual"
)
plot(resid_fly.rename("2s5s10s rolling residual") * 100, which="left")
z_fly = (resid_fly - resid_fly.rolling(65).mean()) / resid_fly.rolling(65).std()
plot(z_fly.rename("z-score (65d)"), which="right",
     indicators=[{"kind": "zbands", "entry": 2, "stop": 3}])
legend(show_date=True, loc="upper left")
plt.show()

## 2. OU calibration + optimal entry/exit bands (spec B)

Calibrate OU on the rolling residual, then compute Zeng-Lee optimal thresholds.

In [ ]:
from RVUtils.mean_reversion import calibrate_ou, optimal_ou_thresholds, ou_sscore, adf_gate

# OU calibration on 2s5s10s rolling residual
ou = calibrate_ou(resid_fly)
print("OU calibration (2s5s10s rolling residual):")
for k in ["mu", "kappa", "sigma", "half_life"]:
    print(f"  {k}: {ou[k]:.4f}")

# Optimal thresholds (in sigma_eq units)
from RVUtils.cost_model import structure_cost_bps
cost_bp = structure_cost_bps([2, 5, 10], [abs(w_2s5s10s["2y"]), abs(w_2s5s10s["5y"]), abs(w_2s5s10s["10y"])])
sigma_eq = ou["sigma"] / np.sqrt(2 * ou["kappa"]) if ou["kappa"] > 0 else np.nan
cost_sigma = (cost_bp / 10000) / sigma_eq if np.isfinite(sigma_eq) and sigma_eq > 0 else 0.0

a_sym, b_sym = optimal_ou_thresholds(ou["kappa"], ou["sigma"], cost=cost_sigma, case="symmetric")
a_lo, b_lo = optimal_ou_thresholds(ou["kappa"], ou["sigma"], cost=cost_sigma, case="long_only")
print(f"\nStructure cost: {cost_bp:.2f} bp -> {cost_sigma:.4f} sigma_eq units")
print(f"Optimal symmetric bands: entry ±{a_sym:.3f}σ, exit ±{-b_sym:.3f}σ")
print(f"Optimal long-only bands: entry {a_lo:.3f}σ, exit {b_lo:.3f}σ")

## 3. OU S-score + ADF gate (spec D, E)

S-score standardizes the residual by the OU equilibrium volatility. ADF gate filters non-stationary residuals.

In [ ]:
# S-score on multiple fly residuals
flies = {
    "2s5s10s": (["2y", "5y", "10y"], w_2s5s10s),
    "5s10s30s": (["5y", "10y", "30y"], fly_w_fn("5y", "10y", "30y")),
    "2s10s30s": (["2y", "10y", "30y"], fly_w_fn("2y", "10y", "30y")),
    "3s7s20s": (["3y", "7y", "20y"], fly_w_fn("3y", "7y", "20y")),
}

rows = []
for name, (legs, wts) in flies.items():
    resid = rolling_residual(df_s, legs, weights=wts, window=200, k=3)
    if resid.notna().sum() < 30:
        continue
    sscore = ou_sscore(resid)
    adf_ok = adf_gate(resid, pval=0.10)
    ou_p = calibrate_ou(resid)
    rows.append({
        "fly": name,
        "s_score": round(float(sscore.iloc[-1]), 3) if sscore.notna().sum() > 0 else np.nan,
        "adf_pass": adf_ok,
        "half_life": round(ou_p["half_life"], 1),
        "kappa": round(ou_p["kappa"], 4),
        "weights": {k: round(v, 3) for k, v in wts.items()},
    })

df_sscore = pd.DataFrame(rows).set_index("fly")
print("S-score + ADF gate for SOFR flies (rolling PCA residual):")
df_sscore

## 4. Eigenportfolio returns (spec F)

Factor-mimicking portfolio returns: `F_j(t) = Σ_i (v_ji / σ_i) R_i(t)`.

In [ ]:
_, get_model_fn = make_pca_rv_builder(df_s, on="levels", n_factors=3, sort_by_tenor=False)[-2:]
# re-fit to get the model
fit2, *_, get_model2 = make_pca_rv_builder(df_s, on="levels", n_factors=3, sort_by_tenor=False)
fit2()
model, _ = get_model2()

asset_returns = df_s.diff().dropna()
asset_vols = asset_returns.std()
F = eigenportfolio_returns(model.loadings[["PC1", "PC2", "PC3"]], asset_returns, asset_vols)

fig, axes = plt.subplots(3, 1, figsize=(15, 10), sharex=True)
for i, pc in enumerate(["PC1", "PC2", "PC3"]):
    axes[i].plot(F[pc].cumsum(), label=f"{pc} eigenportfolio (cumulative)")
    axes[i].legend(loc="upper left")
    axes[i].grid(True)
axes[0].set_title("Eigenportfolio cumulative returns — SOFR curve")
plt.tight_layout()
plt.show()

print("Eigenportfolio return correlation matrix:")
F.corr().round(3)

## 5. Lead-lag — Lévy area, signal, cross-correlation, Granger (spec C)

Which leg leads informs execution timing and hedging variable choice.

In [ ]:
from RVUtils.lead_lag import levy_area, levy_area_signal, xcorr_lead_lag, granger_lead_lag

# Cross-correlation lead-lag
ll = xcorr_lead_lag(df_s["5y"], df_s["10y"], max_lag=5)
print(f"5y vs 10y cross-corr lead-lag: lag={ll['lag']} (positive => 5y leads), corr={ll['corr']:.4f}")

ll2 = xcorr_lead_lag(df_s["2y"], df_s["30y"], max_lag=5)
print(f"2y vs 30y cross-corr lead-lag: lag={ll2['lag']}, corr={ll2['corr']:.4f}")

# Granger causality
gr = granger_lead_lag(df_s["5y"], df_s["10y"], max_lag=5)
print(f"\nGranger causality (5y -> 10y): best lag={gr['lag']}, p-value={gr['pvalue']:.4f}")
gr2 = granger_lead_lag(df_s["10y"], df_s["5y"], max_lag=5)
print(f"Granger causality (10y -> 5y): best lag={gr2['lag']}, p-value={gr2['pvalue']:.4f}")

# Lévy area
la = levy_area(df_s["5y"], df_s["10y"], window=20)

# Lévy area signal (entry/exit with persistence)
la_sig = levy_area_signal(df_s["5y"], df_s["10y"], window=20, theta_entry=1.5, theta_exit=0.5)
print(f"\nLévy area signal: {(la_sig == 1).sum()} long, {(la_sig == -1).sum()} short, {(la_sig == 0).sum()} flat periods")

fig, axes = plt.subplots(2, 1, figsize=(15, 8), sharex=True)
axes[0].plot(la.index, la.values, label="Lévy area (20d)")
axes[0].axhline(0, color="black", lw=0.5)
axes[0].set_title("Lévy area: 5y vs 10y SOFR (>0 ⇒ 5y leads)")
axes[0].legend(loc="upper left")
axes[0].grid(True)

axes[1].plot(la_sig.index, la_sig.values, label="Lévy area signal", color="tab:purple", drawstyle="steps-post")
axes[1].set_yticks([-1, 0, 1])
axes[1].set_yticklabels(["Short (-1)", "Flat (0)", "Long (+1)"])
axes[1].set_title("Lévy area signal (θ_entry=1.5, θ_exit=0.5)")
axes[1].legend(loc="upper left")
axes[1].grid(True)

plt.tight_layout()
plt.show()

## 6. Cost-aware fly screener — `make_pca_fly_rv_screener` (spec G, H, J)

End-to-end *Catching the Butterfly* pipeline: rolling PCA residual → ADF gate → OU half-life → cost-aware composite ranking.

Runs the screener twice — without cost adjustments and with `cost_z` + `lambda_carry` — to show how transaction costs and carry reshape the ranking.

In [ ]:
from RVUtils.screener_rv import make_pca_fly_rv_screener
from RVUtils.cost_model import transaction_cost_bps, structure_cost_bps

structures = {
    "2s5s10s": ("2y", "5y", "10y"),
    "2s5s30s": ("2y", "5y", "30y"),
    "2s10s30s": ("2y", "10y", "30y"),
    "3s5s10s": ("3y", "5y", "10y"),
    "3s7s10s": ("3y", "7y", "10y"),
    "3s7s20s": ("3y", "7y", "20y"),
    "5s7s10s": ("5y", "7y", "10y"),
    "5s10s30s": ("5y", "10y", "30y"),
    "7s10s30s": ("7y", "10y", "30y"),
    "7s20s30s": ("7y", "20y", "30y"),
}

# PCA-neutral weights for each fly
weights_map = {}
for name, (s, b, l) in structures.items():
    try:
        w = fly_w_fn(s, b, l)
        weights_map[name] = w
    except Exception as e:
        print(f"Skipping {name}: {e}")

# --- Cost model demo ---
tenor_map = {"2y": 2, "3y": 3, "5y": 5, "7y": 7, "10y": 10, "20y": 20, "30y": 30}
print("Transaction cost model (bid-ask half-spread, bp):")
for t in ["2y", "5y", "10y", "20y", "30y"]:
    print(f"  {t}: {transaction_cost_bps(tenor_map[t]):.2f} bp")

print("\nStructure costs:")
for name, (s, b, l) in list(structures.items())[:4]:
    wts = weights_map.get(name, {})
    if wts:
        legs = [tenor_map[s], tenor_map[b], tenor_map[l]]
        ws = [abs(wts[s]), abs(wts[b]), abs(wts[l])]
        c = structure_cost_bps(legs, ws)
        print(f"  {name}: {c:.2f} bp")

# --- Screener: no cost adjustment (baseline) ---
result_nocost = make_pca_fly_rv_screener(
    df_s, structures, weights_map,
    window=200, k=3, cost_z=0.0, lambda_carry=0.0, adf_pval=0.10,
)

# --- Screener: with cost deduction ---
result_cost = make_pca_fly_rv_screener(
    df_s, structures, weights_map,
    window=200, k=3, cost_z=0.02, lambda_carry=0.0, adf_pval=0.10,
)

print(f"\n--- Baseline (no cost) ---")
print(result_nocost[["zscore", "half_life", "adf_pass", "composite", "direction"]].to_string(float_format="{:.3f}".format))

print(f"\n--- With cost_z=0.02 deduction ---")
print(result_cost[["zscore", "half_life", "adf_pass", "composite", "direction"]].to_string(float_format="{:.3f}".format))

# Show the composite shift
comparison = pd.DataFrame({
    "composite_baseline": result_nocost["composite"],
    "composite_with_cost": result_cost["composite"].reindex(result_nocost.index),
    "delta": result_cost["composite"].reindex(result_nocost.index).abs() - result_nocost["composite"].abs(),
})
print(f"\nCost impact on |composite| (negative = cost hurts ranking):")
print(comparison.to_string(float_format="{:.4f}".format))

## 7. Deep dive — top-ranked fly

Rolling residual, S-score with Zeng-Lee optimal OU bands, and OU forecast for the top-ranked structure.

In [ ]:
from RVUtils.mean_reversion import ou_conditional

result = result_nocost  # use baseline for deep dive
top_name = result.index[0]
top_legs = list(structures[top_name])
top_wts = weights_map[top_name]
print(f"Top fly: {top_name}  weights: {{{', '.join(f'{k}: {v:.3f}' for k, v in top_wts.items())}}}")
print(f"  direction: {result.loc[top_name, 'direction']}  z: {result.loc[top_name, 'zscore']:.2f}  hl: {result.loc[top_name, 'half_life']:.1f}d  adf: {result.loc[top_name, 'adf_pass']}")

top_resid = rolling_residual(df_s, top_legs, weights=top_wts, window=200, k=3)
top_ou = calibrate_ou(top_resid)
top_sscore = ou_sscore(top_resid)

# OU conditional forecast
x0 = float(top_resid.iloc[-1])
cond = ou_conditional(x0, top_ou, horizon=top_ou["half_life"])
print(f"  OU forecast ({top_ou['half_life']:.0f}d): current={x0*100:.2f}bp -> E[x]={cond['mean']*100:.2f}bp ± {cond['std']*100:.2f}bp")

# Optimal OU bands for this fly
top_legs_tenors = [tenor_map[l] for l in top_legs]
top_legs_abs_w = [abs(top_wts[l]) for l in top_legs]
top_cost_bp = structure_cost_bps(top_legs_tenors, top_legs_abs_w)
top_sigma_eq = top_ou["sigma"] / np.sqrt(2 * top_ou["kappa"]) if top_ou["kappa"] > 0 else np.nan
top_cost_sigma = (top_cost_bp / 10000) / top_sigma_eq if np.isfinite(top_sigma_eq) and top_sigma_eq > 0 else 0.0

a_opt, b_opt = optimal_ou_thresholds(top_ou["kappa"], top_ou["sigma"], cost=top_cost_sigma, case="symmetric")
print(f"  Zeng-Lee optimal bands: entry ±{a_opt:.3f}σ_eq, exit ±{-b_opt:.3f}σ_eq (cost={top_cost_bp:.2f}bp = {top_cost_sigma:.4f}σ_eq)")

fig, axes = plt.subplots(2, 1, figsize=(15, 9), sharex=True)
axes[0].plot(top_resid.index, top_resid.values * 100, label=f"{top_name} rolling PCA residual (bp)")
axes[0].axhline(top_ou["mu"] * 100, color="red", ls="--", label=f"OU μ = {top_ou['mu']*100:.2f}bp")
axes[0].legend(loc="upper left")
axes[0].grid(True)
axes[0].set_title(f"{top_name}: rolling PCA residual + OU mean")

axes[1].plot(top_sscore.index, top_sscore.values, label="S-score", color="tab:orange")

# Avellaneda-Lee fixed thresholds
axes[1].axhline(1.75, color="green", ls="--", alpha=0.5, label="A-L entry ±1.75")
axes[1].axhline(-1.75, color="green", ls="--", alpha=0.5)
axes[1].axhline(0.75, color="gray", ls=":", alpha=0.5, label="A-L exit ±0.75")
axes[1].axhline(-0.75, color="gray", ls=":", alpha=0.5)

# Zeng-Lee optimal bands (in S-score units — a_opt is in sigma_eq, S-score IS in sigma_eq)
axes[1].axhline(a_opt, color="blue", ls="-", lw=1.5, alpha=0.8, label=f"ZL optimal entry ±{a_opt:.2f}")
axes[1].axhline(-a_opt, color="blue", ls="-", lw=1.5, alpha=0.8)
axes[1].axhline(-b_opt, color="cornflowerblue", ls="--", lw=1.5, alpha=0.8, label=f"ZL optimal exit ±{-b_opt:.2f}")
axes[1].axhline(b_opt, color="cornflowerblue", ls="--", lw=1.5, alpha=0.8)

axes[1].axhline(0, color="black", lw=0.5)
axes[1].legend(loc="upper left", fontsize="small")
axes[1].grid(True)
axes[1].set_title(f"{top_name}: S-score with Avellaneda-Lee (dashed) and Zeng-Lee optimal (solid) bands")

plt.tight_layout()
plt.show()

## 8. OLS sub-period beta stability (spec I)

Check whether the PCA fly's beta to level/slope has been stable across regimes.

In [ ]:
from RVUtils.regression import ols_segment

# Build a fly timeseries and regress on level (PC1 score) and slope (PC2 score)
_, scores = get_model2()
fly_ts = sum(top_wts[c] * df_s[c] for c in top_legs).dropna()
fly_ts.name = top_name

common = fly_ts.index.intersection(scores.index)
mid = common[len(common) // 2]
periods = [(common[0], mid), (mid, common[-1])]

seg = ols_segment(
    fly_ts.loc[common],
    scores[["PC1", "PC2"]].loc[common],
    periods,
)

print(f"Beta stability for {top_name} vs PC1/PC2:")
for s in seg:
    p0, p1 = pd.Timestamp(s["period"][0]), pd.Timestamp(s["period"][1])
    print(f"  {p0.date()} -> {p1.date()}: "
          f"β_PC1={s['betas']['PC1']:.4f}, β_PC2={s['betas']['PC2']:.4f}, "
          f"adj-R²={s['adj_r2']:.3f} (n={s['nobs']})")